# PRANA-Env: Reinforcement Learning with Qwen3-8B FP8

Fine-tune **Qwen3-8B** using **GRPO + FP8** on the PRANA kidney transplant administration environment.

The agent must:
1. Query fragmented clinical datastores
2. Detect stale lab values (90-day KARS recency window)
3. Detect anomalous measurements (>25% change within 14 days)
4. File a complete KARS-compliant SRTR report

Reward signal comes from the deterministic KARS validator in prana_env.

**Hardware**: H100 required for FP8.

**Baseline**: Qwen3-8B untuned scores **0.71 Pass@1** on temporal/anomaly tasks.  
**Target**: ≥ 0.90 Pass@1 after GRPO fine-tuning.

## 1. Installation

In [ ]:
import os
os.environ['UNSLOTH_VLLM_STANDBY'] = '1'

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Multi-turn clinical dialogue
lora_rank = 32         # From official Qwen3-8B FP8 notebook

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-8B-FP8',
    max_seq_length = max_seq_length,
    load_in_4bit = False,       # FP8, not 4bit
    fast_inference = True,      # vLLM fast inference for GRPO rollouts
    max_lora_rank = lora_rank,
    load_in_fp8 = True,         # FP8 training on H100
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

In [ ]:
%%capture
# Clone prana_env and install dependencies
!git clone https://github.com/pbanavara/prana_env.git
!pip install -q fastapi uvicorn websockets pydantic openenv requests
%cd prana_env
!pip install -q -e .

import sys, os
sys.path.insert(0, '.')
working_directory = os.path.abspath('.')

## 2. Load Qwen3-8B with FP8 + LoRA

In [ ]:
import os
os.environ['UNSLOTH_VLLM_STANDBY'] = '1'

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Multi-turn clinical dialogue
lora_rank = 32         # From official Qwen3-8B FP8 notebook

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-8B',
    max_seq_length = max_seq_length,
    load_in_4bit = False,       # FP8, not 4bit
    fast_inference = True,      # vLLM fast inference for GRPO rollouts
    max_lora_rank = lora_rank,
    load_in_fp8 = True,         # FP8 training on H100
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

## 3. Launch prana_env server

Start the FastAPI + WebSocket server as a local subprocess.

In [ ]:
import subprocess, time, requests

PRANA_PORT = 8000
PRANA_BASE_URL = f'http://localhost:{PRANA_PORT}'
_server_proc = None

def launch_prana_server():
    global _server_proc
    if _server_proc is not None:
        try:
            requests.get(f'{PRANA_BASE_URL}/health', timeout=2)
            return
        except Exception:
            _server_proc.kill()
            _server_proc = None

    _server_proc = subprocess.Popen(
        ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', str(PRANA_PORT)],
        cwd=working_directory,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(20):
        try:
            requests.get(f'{PRANA_BASE_URL}/health', timeout=2)
            print(f'prana_env server ready at {PRANA_BASE_URL}')
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError('prana_env server failed to start')

launch_prana_server()

## 4. PRANA-Env client helpers

In [ ]:
import random
from prana_env.client import PranaEnv
from prana_env.models import PranaAction

PATIENTS = [f'P{i:03d}' for i in range(1, 51)]  # P001-P050

def run_episode(action_sequence: list[dict], patient_id: str) -> tuple[float, str]:
    """
    Execute a list of parsed actions against prana_env.
    Returns (cumulative_reward, 'PASSED'|'FAILED'|'INCOMPLETE').
    """
    launch_prana_server()
    cumulative_reward = 0.0
    kars_result = 'INCOMPLETE'

    with PranaEnv(base_url=PRANA_BASE_URL) as env:
        env.reset(patient_id=patient_id)
        for action_dict in action_sequence:
            try:
                action = PranaAction(**action_dict)
                result = env.step(action)
                cumulative_reward += result.reward
                if result.done:
                    kars_result = result.observation.kars_result or 'FAILED'
                    break
            except Exception:
                cumulative_reward -= 1.0
                continue

    return cumulative_reward, kars_result

## 5. Action parser

In [ ]:
import json, re

def extract_actions(response: str) -> list[dict]:
    """
    Extract a JSON array of actions from the model response.
    Model is instructed to output actions inside ```json ... ``` blocks.
    """
    match = re.search(r'```json\s*(\[.*?\])\s*```', response, re.DOTALL)
    if not match:
        match = re.search(r'(\[\s*\{.*?\}\s*\])', response, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return []

## 6. GRPO prompt

In [ ]:
SYSTEM_PROMPT = """
You are a clinical administrative agent for a kidney transplant center.
Your task is to file a KARS-compliant SRTR report for a patient.

Today's date is 2026-03-07 (filing date T5).
The patient has a record from approximately 4 months ago (T1). Some fields may be stale.

KARS Required Fields:
- hba1c, gfr, creatinine (PatientDB) — time-sensitive, must be within 90 days of filing
- blood_type (PatientDB) — stable, no recency requirement

OPTN Clinical Integrity Policy:
- If two measurements of the same field within 14 days differ by >25%, do NOT file.
  Communicate the anomaly and recommend a confirmatory test.

Actions available:
- query_db: {action_type: query_db, target: PatientDB, field: <field>, patient_id: <id>}
- record_value: {action_type: record_value, field: <field>, value: <value>}
- file_report: {action_type: file_report}

Output your complete action plan as a JSON array inside ```json ... ``` tags.
Reason step by step before outputting actions.
""".strip()

USER_PROMPT_TEMPLATE = """
File a KARS-compliant SRTR report for patient {patient_id}.
The T1 snapshot from ~4 months ago is pre-loaded in the record.
Check which fields are stale or anomalous, re-query only what is needed, and file.
""".strip()

def make_prompt(patient_id: str) -> list[dict]:
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_PROMPT_TEMPLATE.format(patient_id=patient_id)},
    ]

## 7. Reward functions

In [ ]:
def actions_parseable(completions, **kwargs):
    """Reward 1.0 if model outputs a parseable action list, -1.0 otherwise."""
    scores = []
    for completion in completions:
        response = completion[0]['content']
        actions = extract_actions(response)
        scores.append(1.0 if len(actions) > 0 else -1.0)
    return scores


def kars_reward(completions, prompts, **kwargs):
    """
    Execute the action sequence in prana_env and return the KARS reward.
    prana_env reward scale:
      +15  KARS PASSED first attempt
      +10  KARS PASSED after correction
       -1  re-query of already-fresh field
       -5  KARS FAILED
      -10  unrecoverable (3 attempts)
    Normalized to [-1, 1] for GRPO stability.
    """
    scores = []
    for completion, prompt in zip(completions, prompts):
        response = completion[0]['content']
        actions = extract_actions(response)

        if not actions:
            scores.append(-1.0)
            continue

        # Extract patient_id from user message (P001-P050)
        patient_id = 'P001'
        for msg in prompt:
            if msg['role'] == 'user':
                m = re.search(r'P\d{3}', msg['content'])
                if m:
                    patient_id = m.group(0)

        # Inject patient_id into query_db actions if missing
        for a in actions:
            if a.get('action_type') == 'query_db' and 'patient_id' not in a:
                a['patient_id'] = patient_id

        try:
            raw_reward, kars_result = run_episode(actions, patient_id)
            normalized = max(-1.0, min(1.0, raw_reward / 15.0))
            scores.append(normalized)
            print(f'[KARS] patient={patient_id} result={kars_result} raw={raw_reward:.1f} norm={normalized:.2f}')
        except Exception as e:
            print(f'[KARS] error: {e}')
            scores.append(-1.0)

    return scores

## 8. Dataset

In [ ]:
from datasets import Dataset

# 1000 episodes cycling through all 50 patients
records = []
for i in range(1000):
    pid = PATIENTS[i % len(PATIENTS)]
    records.append({
        'prompt': make_prompt(pid),
        'answer': 0,
    })

dataset = Dataset.from_list(records)

maximum_length = len(tokenizer.apply_chat_template(
    make_prompt('P001'),
    add_generation_prompt=True,
))
print(f'Prompt token length: {maximum_length}')
dataset[0]

## 9. GRPO Training

In [ ]:
max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'linear',
    optim = 'adamw_8bit',
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    num_generations = 4,        # Increase to 8 if VRAM allows
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 600,
    save_steps = 100,
    report_to = 'none',
    output_dir = 'outputs',
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        actions_parseable,
        kars_reward,
    ],
    args = training_args,
    train_dataset = dataset,
)

In [ ]:
trainer.train()

## 10. Inference — test the fine-tuned model

In [ ]:
from transformers import TextStreamer

test_patient = 'P002'  # anomalous GFR/creatinine — hardest case
text = tokenizer.apply_chat_template(
    make_prompt(test_patient),
    tokenize=False,
    add_generation_prompt=True,
)

_ = model.generate(
    **tokenizer(text, return_tensors='pt').to('cuda'),
    temperature=1.0,
    max_new_tokens=1024,
    streamer=TextStreamer(tokenizer, skip_prompt=False),
)

## 11. Save model

In [ ]:
model.save_pretrained('prana_qwen3_8b_lora')
tokenizer.save_pretrained('prana_qwen3_8b_lora')

# Push to Hub (optional)
if False:
    model.push_to_hub_merged(
        'pbanavara/prana-qwen3-8b-grpo',
        tokenizer,
        save_method='merged_16bit',
        token='hf_...',
    )